# Notional damage scenarios (PBM)

This notebook runs three **notional initial-damage scenarios** on a 3D lattice and an undamaged scenario. Every case uses the same base seed and realization-index streams, providing common random numbers for the scenario comparison.

Scenarios (lower storey, z=0):
0. No initial damage
1. Corner node loss
2. Periphery (edge-middle) node loss
3. Internal (central) node loss


## Probability of failure propagation = percolation threshold = 0.67

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
MC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {MC_WORKERS} workers.")

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_damage_scenarios.db'
OUT_DB = REPO_ROOT / 'output' / 'results_damage_scenarios_pe67e-2.db'
FIGS   = REPO_ROOT / 'output' / 'figs_damage_scenarios_pe67e-2'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [1] Parameters ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.67

SAMPLES   = 1_000_000
THRESHOLD = 2
SEED      = 123
LOG_ATTEMPTS = 'no'  # 'yes' or 'no', must be 'no' if used with parallel computing
PLOT_MODE = 'standard'  # use 'publication' for external-LaTeX export

# Notional initial-damage scenarios (x,y,z) on lower storey (z=0)
SCENARIOS = {
    0: ('no damage',                    None),
    1: ('Corner (0,0,0)',               (0, 0, 0)),
    2: ('Periphery (1,0,0)',   (1, 0, 0)),
    3: ('Internal (1,1,0)',      (1, 1, 0)),
}

if OUT_DB.exists():
    raise FileExistsError(f'Archive existing results or choose a fresh OUT_DB before running: {OUT_DB}')

# --- [2] Build lattice input DB ---
from pbm.network.builder import build_and_insert

build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
import sqlite3

with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run the four scenarios (writes 4 runs into OUT_DB) ---
import cli.cli_simulation as cli_sim

for scenario_id, (label, coords) in SCENARIOS.items():
    print(f"\nRunning scenario {scenario_id}: {label}")

    argv = [
        "cli_simulation.py",
        "--db", str(IN_DB),
        "--outdb", str(OUT_DB),
        "--scenario-id", str(scenario_id),
        "--samples", str(SAMPLES),
        "--threshold", str(THRESHOLD),
        "--seed", str(SEED),
        "--log-attempts", str(LOG_ATTEMPTS),
        "--mc-workers", str(MC_WORKERS),
    ]

    # Only add notional-damage input when coords is provided
    if coords is not None:
        x, y, z = coords
        argv += ["--damage-coords", f"{x},{y},{z}"]

    sys.argv = argv
    cli_sim.main()

print('\nAll scenarios finished.')
print('OUT_DB:', OUT_DB)

# --- [5a] Post-processing plots ---
import sqlite3
import pandas as pd
import sys
import cli.cli_post_processing as cli_post

with sqlite3.connect(OUT_DB) as conn:
    runs = pd.read_sql("SELECT id AS run_id FROM runs ORDER BY id", conn)

for run_id in runs["run_id"]:
    sys.argv = [
        "cli_post_processing.py",
        "--outdb", str(OUT_DB),
        "--figs-root", str(FIGS),
        "--make-plots", "yes",
        "--plot-mode", PLOT_MODE,
        "--run-id", str(int(run_id)),
    ]
    cli_post.main()

print("Figures written to:", FIGS)

# --- [5b] Entropy-based robustness index (pipeline function) ---

from pathlib import Path
import sqlite3
from pbm.post.robustness_plots import plot_entropy_robustness_index

# Paths (as defined above in the notebook)
outdb = OUT_DB
figs_dir = FIGS
figs_dir.mkdir(parents=True, exist_ok=True)

# Failure event definition(s)
S_LIM_VALUES = [2, 16]  # Manuscript adverse states; extend for additional analyses.

# --- Scenario-based run selection (UI layer) ---
BASELINE_SCENARIO = 0

# All non-baseline scenarios
SCENARIO_KEYS = sorted(k for k in SCENARIOS if k != BASELINE_SCENARIO)

# Resolve run IDs from stored scenario IDs; do not assume insertion order.
with sqlite3.connect(outdb) as conn:
    run_rows = conn.execute(
        "SELECT id, scenario_id FROM runs ORDER BY id;"
    ).fetchall()
SCENARIO_TO_RUN_ID = {}
for run_id, scenario_id in run_rows:
    if scenario_id in SCENARIO_TO_RUN_ID:
        raise RuntimeError(f"Multiple runs found for scenario_id={scenario_id}.")
    SCENARIO_TO_RUN_ID[int(scenario_id)] = int(run_id)

# Derived DB-facing values (used by pipeline)
BASELINE_RUN_ID = SCENARIO_TO_RUN_ID[BASELINE_SCENARIO]
SCENARIO_RUN_IDS = [SCENARIO_TO_RUN_ID[k] for k in SCENARIO_KEYS]

# Legend labels (DB run_id -> human label)
scenario_labels = {
    SCENARIO_TO_RUN_ID[k]: SCENARIOS[k][0]
    for k in SCENARIO_KEYS
}

with sqlite3.connect(outdb) as conn:
    for s_lim in S_LIM_VALUES:
        plot_entropy_robustness_index(
            conn,
            baseline_run_id=BASELINE_RUN_ID,
            damaged_run_ids=SCENARIO_RUN_IDS,
            s_lim=int(s_lim),
            figs_dir=figs_dir,
            filename=f"entropy_robustness_index_Slim_{int(s_lim)}",
            formats=("pdf",),
            pi_min=-3.0,
            pi_max=3.0,
            n_grid=1001,
            scenario_labels=scenario_labels,
            coherence_mode="strict",
            mc_tolerance=0.0,
            plot_mode=PLOT_MODE
        )

print(f"Saved entropy robustness figures to: {figs_dir.resolve()}")


## Probability of failure propagation 0.4

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
MC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {MC_WORKERS} workers.")

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_damage_scenarios.db'
OUT_DB = REPO_ROOT / 'output' / 'results_damage_scenarios_pe40e-2.db'
FIGS   = REPO_ROOT / 'output' / 'figs_damage_scenarios_pe40e-2'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [1] Parameters ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.4

SAMPLES   = 1_000_000
THRESHOLD = 2
SEED      = 123
LOG_ATTEMPTS = 'no'  # 'yes' or 'no', must be 'no' if used with parallel computing
PLOT_MODE = 'standard'  # use 'publication' for external-LaTeX export

# Notional initial-damage scenarios (x,y,z) on lower storey (z=0)
SCENARIOS = {
    0: ('no damage',                    None),
    1: ('Corner (0,0,0)',               (0, 0, 0)),
    2: ('Periphery (1,0,0)',   (1, 0, 0)),
    3: ('Internal (1,1,0)',      (1, 1, 0)),
}

if OUT_DB.exists():
    raise FileExistsError(f'Archive existing results or choose a fresh OUT_DB before running: {OUT_DB}')

# --- [2] Build lattice input DB ---
from pbm.network.builder import build_and_insert

build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
import sqlite3

with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run the four scenarios (writes 4 runs into OUT_DB) ---
import cli.cli_simulation as cli_sim

for scenario_id, (label, coords) in SCENARIOS.items():
    print(f"\nRunning scenario {scenario_id}: {label}")

    argv = [
        "cli_simulation.py",
        "--db", str(IN_DB),
        "--outdb", str(OUT_DB),
        "--scenario-id", str(scenario_id),
        "--samples", str(SAMPLES),
        "--threshold", str(THRESHOLD),
        "--seed", str(SEED),
        "--log-attempts", str(LOG_ATTEMPTS),
        "--mc-workers", str(MC_WORKERS),
    ]

    # Only add notional-damage input when coords is provided
    if coords is not None:
        x, y, z = coords
        argv += ["--damage-coords", f"{x},{y},{z}"]

    sys.argv = argv
    cli_sim.main()

print('\nAll scenarios finished.')
print('OUT_DB:', OUT_DB)

# --- [5a] Post-processing plots ---
import sqlite3
import pandas as pd
import sys
import cli.cli_post_processing as cli_post

with sqlite3.connect(OUT_DB) as conn:
    runs = pd.read_sql("SELECT id AS run_id FROM runs ORDER BY id", conn)

for run_id in runs["run_id"]:
    sys.argv = [
        "cli_post_processing.py",
        "--outdb", str(OUT_DB),
        "--figs-root", str(FIGS),
        "--make-plots", "yes",
        "--plot-mode", PLOT_MODE,
        "--run-id", str(int(run_id)),
    ]
    cli_post.main()

print("Figures written to:", FIGS)

# --- [5b] Entropy-based robustness index (pipeline function) ---

from pathlib import Path
import sqlite3
from pbm.post.robustness_plots import plot_entropy_robustness_index

# Paths (as defined above in the notebook)
outdb = OUT_DB
figs_dir = FIGS
figs_dir.mkdir(parents=True, exist_ok=True)

# Failure event definition(s)
S_LIM_VALUES = [2, 16]  # Manuscript adverse states; extend for additional analyses.

# --- Scenario-based run selection (UI layer) ---
BASELINE_SCENARIO = 0

# All non-baseline scenarios
SCENARIO_KEYS = sorted(k for k in SCENARIOS if k != BASELINE_SCENARIO)

# Resolve run IDs from stored scenario IDs; do not assume insertion order.
with sqlite3.connect(outdb) as conn:
    run_rows = conn.execute(
        "SELECT id, scenario_id FROM runs ORDER BY id;"
    ).fetchall()
SCENARIO_TO_RUN_ID = {}
for run_id, scenario_id in run_rows:
    if scenario_id in SCENARIO_TO_RUN_ID:
        raise RuntimeError(f"Multiple runs found for scenario_id={scenario_id}.")
    SCENARIO_TO_RUN_ID[int(scenario_id)] = int(run_id)

# Derived DB-facing values (used by pipeline)
BASELINE_RUN_ID = SCENARIO_TO_RUN_ID[BASELINE_SCENARIO]
SCENARIO_RUN_IDS = [SCENARIO_TO_RUN_ID[k] for k in SCENARIO_KEYS]

# Legend labels (DB run_id -> human label)
scenario_labels = {
    SCENARIO_TO_RUN_ID[k]: SCENARIOS[k][0]
    for k in SCENARIO_KEYS
}

with sqlite3.connect(outdb) as conn:
    for s_lim in S_LIM_VALUES:
        plot_entropy_robustness_index(
            conn,
            baseline_run_id=BASELINE_RUN_ID,
            damaged_run_ids=SCENARIO_RUN_IDS,
            s_lim=int(s_lim),
            figs_dir=figs_dir,
            filename=f"entropy_robustness_index_Slim_{int(s_lim)}",
            formats=("pdf",),
            pi_min=-3.0,
            pi_max=3.0,
            n_grid=1001,
            scenario_labels=scenario_labels,
            coherence_mode="strict",
            mc_tolerance=0.0,
            plot_mode=PLOT_MODE
        )

print(f"Saved entropy robustness figures to: {figs_dir.resolve()}")


## Probability of failure propagation = percolation threshold = 0.2

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
MC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {MC_WORKERS} workers.")

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_damage_scenarios.db'
OUT_DB = REPO_ROOT / 'output' / 'results_damage_scenarios_pe20e-2.db'
FIGS   = REPO_ROOT / 'output' / 'figs_damage_scenarios_pe20e-2'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [1] Parameters ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.2

SAMPLES   = 1_000_000
THRESHOLD = 2
SEED      = 123
LOG_ATTEMPTS = 'no'  # 'yes' or 'no', must be 'no' if used with parallel computing
PLOT_MODE = 'standard'  # use 'publication' for external-LaTeX export

# Notional initial-damage scenarios (x,y,z) on lower storey (z=0)
SCENARIOS = {
    0: ('no damage',                    None),
    1: ('Corner (0,0,0)',               (0, 0, 0)),
    2: ('Periphery (1,0,0)',   (1, 0, 0)),
    3: ('Internal (1,1,0)',      (1, 1, 0)),
}

if OUT_DB.exists():
    raise FileExistsError(f'Archive existing results or choose a fresh OUT_DB before running: {OUT_DB}')

# --- [2] Build lattice input DB ---
from pbm.network.builder import build_and_insert

build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
import sqlite3

with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run the four scenarios (writes 4 runs into OUT_DB) ---
import cli.cli_simulation as cli_sim

for scenario_id, (label, coords) in SCENARIOS.items():
    print(f"\nRunning scenario {scenario_id}: {label}")

    argv = [
        "cli_simulation.py",
        "--db", str(IN_DB),
        "--outdb", str(OUT_DB),
        "--scenario-id", str(scenario_id),
        "--samples", str(SAMPLES),
        "--threshold", str(THRESHOLD),
        "--seed", str(SEED),
        "--log-attempts", str(LOG_ATTEMPTS),
        "--mc-workers", str(MC_WORKERS),
    ]

    # Only add notional-damage input when coords is provided
    if coords is not None:
        x, y, z = coords
        argv += ["--damage-coords", f"{x},{y},{z}"]

    sys.argv = argv
    cli_sim.main()

print('\nAll scenarios finished.')
print('OUT_DB:', OUT_DB)

# --- [5a] Post-processing plots ---
import sqlite3
import pandas as pd
import sys
import cli.cli_post_processing as cli_post

with sqlite3.connect(OUT_DB) as conn:
    runs = pd.read_sql("SELECT id AS run_id FROM runs ORDER BY id", conn)

for run_id in runs["run_id"]:
    sys.argv = [
        "cli_post_processing.py",
        "--outdb", str(OUT_DB),
        "--figs-root", str(FIGS),
        "--make-plots", "yes",
        "--plot-mode", PLOT_MODE,
        "--run-id", str(int(run_id)),
    ]
    cli_post.main()

print("Figures written to:", FIGS)

# --- [5b] Entropy-based robustness index (pipeline function) ---

from pathlib import Path
import sqlite3
from pbm.post.robustness_plots import plot_entropy_robustness_index

# Paths (as defined above in the notebook)
outdb = OUT_DB
figs_dir = FIGS
figs_dir.mkdir(parents=True, exist_ok=True)

# Failure event definition(s)
S_LIM_VALUES = [2, 16]  # Manuscript adverse states; extend for additional analyses.

# --- Scenario-based run selection (UI layer) ---
BASELINE_SCENARIO = 0

# All non-baseline scenarios
SCENARIO_KEYS = sorted(k for k in SCENARIOS if k != BASELINE_SCENARIO)

# Resolve run IDs from stored scenario IDs; do not assume insertion order.
with sqlite3.connect(outdb) as conn:
    run_rows = conn.execute(
        "SELECT id, scenario_id FROM runs ORDER BY id;"
    ).fetchall()
SCENARIO_TO_RUN_ID = {}
for run_id, scenario_id in run_rows:
    if scenario_id in SCENARIO_TO_RUN_ID:
        raise RuntimeError(f"Multiple runs found for scenario_id={scenario_id}.")
    SCENARIO_TO_RUN_ID[int(scenario_id)] = int(run_id)

# Derived DB-facing values (used by pipeline)
BASELINE_RUN_ID = SCENARIO_TO_RUN_ID[BASELINE_SCENARIO]
SCENARIO_RUN_IDS = [SCENARIO_TO_RUN_ID[k] for k in SCENARIO_KEYS]

# Legend labels (DB run_id -> human label)
scenario_labels = {
    SCENARIO_TO_RUN_ID[k]: SCENARIOS[k][0]
    for k in SCENARIO_KEYS
}

with sqlite3.connect(outdb) as conn:
    for s_lim in S_LIM_VALUES:
        plot_entropy_robustness_index(
            conn,
            baseline_run_id=BASELINE_RUN_ID,
            damaged_run_ids=SCENARIO_RUN_IDS,
            s_lim=int(s_lim),
            figs_dir=figs_dir,
            filename=f"entropy_robustness_index_Slim_{int(s_lim)}",
            formats=("pdf",),
            pi_min=-3.0,
            pi_max=3.0,
            n_grid=1001,
            scenario_labels=scenario_labels,
            coherence_mode="strict",
            mc_tolerance=0.0,
            plot_mode=PLOT_MODE
        )

print(f"Saved entropy robustness figures to: {figs_dir.resolve()}")


## Probability of failure propagation = numerical risk-equivalent probability = 0.0927

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
MC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {MC_WORKERS} workers.")

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_damage_scenarios.db'
OUT_DB = REPO_ROOT / 'output' / 'results_damage_scenarios_pe93e-3.db'
FIGS   = REPO_ROOT / 'output' / 'figs_damage_scenarios_pe93e-3'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [1] Parameters ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.0927

SAMPLES   = 1_000_000
THRESHOLD = 2
SEED      = 123
LOG_ATTEMPTS = 'no'  # 'yes' or 'no', must be 'no' if used with parallel computing
PLOT_MODE = 'standard'  # use 'publication' for external-LaTeX export

# Notional initial-damage scenarios (x,y,z) on lower storey (z=0)
SCENARIOS = {
    0: ('no damage',                    None),
    1: ('Corner (0,0,0)',               (0, 0, 0)),
    2: ('Periphery (1,0,0)',   (1, 0, 0)),
    3: ('Internal (1,1,0)',      (1, 1, 0)),
}

if OUT_DB.exists():
    raise FileExistsError(f'Archive existing results or choose a fresh OUT_DB before running: {OUT_DB}')

# --- [2] Build lattice input DB ---
from pbm.network.builder import build_and_insert

build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
import sqlite3

with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run the four scenarios (writes 4 runs into OUT_DB) ---
import cli.cli_simulation as cli_sim

for scenario_id, (label, coords) in SCENARIOS.items():
    print(f"\nRunning scenario {scenario_id}: {label}")

    argv = [
        "cli_simulation.py",
        "--db", str(IN_DB),
        "--outdb", str(OUT_DB),
        "--scenario-id", str(scenario_id),
        "--samples", str(SAMPLES),
        "--threshold", str(THRESHOLD),
        "--seed", str(SEED),
        "--log-attempts", str(LOG_ATTEMPTS),
        "--mc-workers", str(MC_WORKERS),
    ]

    # Only add notional-damage input when coords is provided
    if coords is not None:
        x, y, z = coords
        argv += ["--damage-coords", f"{x},{y},{z}"]

    sys.argv = argv
    cli_sim.main()

print('\nAll scenarios finished.')
print('OUT_DB:', OUT_DB)

# --- [5a] Post-processing plots ---
import sqlite3
import pandas as pd
import sys
import cli.cli_post_processing as cli_post

with sqlite3.connect(OUT_DB) as conn:
    runs = pd.read_sql("SELECT id AS run_id FROM runs ORDER BY id", conn)

for run_id in runs["run_id"]:
    sys.argv = [
        "cli_post_processing.py",
        "--outdb", str(OUT_DB),
        "--figs-root", str(FIGS),
        "--make-plots", "yes",
        "--plot-mode", PLOT_MODE,
        "--run-id", str(int(run_id)),
    ]
    cli_post.main()

print("Figures written to:", FIGS)

# --- [5b] Entropy-based robustness index (pipeline function) ---

from pathlib import Path
import sqlite3
from pbm.post.robustness_plots import plot_entropy_robustness_index

# Paths (as defined above in the notebook)
outdb = OUT_DB
figs_dir = FIGS
figs_dir.mkdir(parents=True, exist_ok=True)

# Failure event definition(s)
S_LIM_VALUES = [2]  # Manuscript adverse states; extend for additional analyses.

# --- Scenario-based run selection (UI layer) ---
BASELINE_SCENARIO = 0

# All non-baseline scenarios
SCENARIO_KEYS = sorted(k for k in SCENARIOS if k != BASELINE_SCENARIO)

# Resolve run IDs from stored scenario IDs; do not assume insertion order.
with sqlite3.connect(outdb) as conn:
    run_rows = conn.execute(
        "SELECT id, scenario_id FROM runs ORDER BY id;"
    ).fetchall()
SCENARIO_TO_RUN_ID = {}
for run_id, scenario_id in run_rows:
    if scenario_id in SCENARIO_TO_RUN_ID:
        raise RuntimeError(f"Multiple runs found for scenario_id={scenario_id}.")
    SCENARIO_TO_RUN_ID[int(scenario_id)] = int(run_id)

# Derived DB-facing values (used by pipeline)
BASELINE_RUN_ID = SCENARIO_TO_RUN_ID[BASELINE_SCENARIO]
SCENARIO_RUN_IDS = [SCENARIO_TO_RUN_ID[k] for k in SCENARIO_KEYS]

# Legend labels (DB run_id -> human label)
scenario_labels = {
    SCENARIO_TO_RUN_ID[k]: SCENARIOS[k][0]
    for k in SCENARIO_KEYS
}

with sqlite3.connect(outdb) as conn:
    for s_lim in S_LIM_VALUES:
        plot_entropy_robustness_index(
            conn,
            baseline_run_id=BASELINE_RUN_ID,
            damaged_run_ids=SCENARIO_RUN_IDS,
            s_lim=int(s_lim),
            figs_dir=figs_dir,
            filename=f"entropy_robustness_index_Slim_{int(s_lim)}",
            formats=("pdf",),
            pi_min=-3.0,
            pi_max=3.0,
            n_grid=1001,
            scenario_labels=scenario_labels,
            coherence_mode="strict",
            mc_tolerance=0.0,
            plot_mode=PLOT_MODE
        )

print(f"Saved entropy robustness figures to: {figs_dir.resolve()}")


## Probability of failure propagation = analytical risk-equivalent probability = 0.094

In [ ]:
# --- [0] Path setup (run first) ---
from __future__ import annotations

import sys
from pathlib import Path
import os

# Number of logical CPUs available to Python
N_CORES = os.cpu_count() or 1

# Reserve two logical CPUs when possible (minimum one worker)
MC_WORKERS = max(1, N_CORES - 2)

print(f"Detected {N_CORES} logical cores, using {MC_WORKERS} workers.")

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'pbm').is_dir() else NB_DIR.parent

if not (REPO_ROOT / 'pbm').exists():
    raise RuntimeError(f"Couldn't find 'pbm/' at {REPO_ROOT}. Run the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT))

IN_DB  = REPO_ROOT / 'input' / 'input_damage_scenarios.db'
OUT_DB = REPO_ROOT / 'output' / 'results_damage_scenarios_pe94e-3.db'
FIGS   = REPO_ROOT / 'output' / 'figs_damage_scenarios_pe94e-3'

IN_DB.parent.mkdir(exist_ok=True)
OUT_DB.parent.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('IN_DB   :', IN_DB)
print('OUT_DB  :', OUT_DB)
print('FIGS    :', FIGS)

# --- [1] Parameters ---
NX, NY, NZ = 4,4,4

PF_NODE = 1e-4
PE_PROP = 0.094

SAMPLES   = 1_000_000
THRESHOLD = 2
SEED      = 123
LOG_ATTEMPTS = 'no'  # 'yes' or 'no', must be 'no' if used with parallel computing
PLOT_MODE = 'standard'  # use 'publication' for external-LaTeX export

# Notional initial-damage scenarios (x,y,z) on lower storey (z=0)
SCENARIOS = {
    0: ('no damage',                    None),
    1: ('Corner (0,0,0)',               (0, 0, 0)),
    2: ('Periphery (1,0,0)',   (1, 0, 0)),
    3: ('Internal (1,1,0)',      (1, 1, 0)),
}

if OUT_DB.exists():
    raise FileExistsError(f'Archive existing results or choose a fresh OUT_DB before running: {OUT_DB}')

# --- [2] Build lattice input DB ---
from pbm.network.builder import build_and_insert

build_and_insert(nx=NX, ny=NY, nz=NZ, db_path=IN_DB, truncate=True)
print('Built input DB.')

# --- [3] Set probabilities in input DB ---
import sqlite3

with sqlite3.connect(IN_DB) as conn:
    conn.execute('PRAGMA foreign_keys=ON;')
    conn.execute('UPDATE nodes SET pf_node = ?;', (float(PF_NODE),))
    conn.execute('UPDATE edges SET pe = ?, pe_fwd = NULL, pe_rev = NULL, directed = 0, blocked = 0;', (float(PE_PROP),))
    conn.commit()

print(f'Updated pf_node={PF_NODE}, pe={PE_PROP}')

# --- [4] Run the four scenarios (writes 4 runs into OUT_DB) ---
import cli.cli_simulation as cli_sim

for scenario_id, (label, coords) in SCENARIOS.items():
    print(f"\nRunning scenario {scenario_id}: {label}")

    argv = [
        "cli_simulation.py",
        "--db", str(IN_DB),
        "--outdb", str(OUT_DB),
        "--scenario-id", str(scenario_id),
        "--samples", str(SAMPLES),
        "--threshold", str(THRESHOLD),
        "--seed", str(SEED),
        "--log-attempts", str(LOG_ATTEMPTS),
        "--mc-workers", str(MC_WORKERS),
    ]

    # Only add notional-damage input when coords is provided
    if coords is not None:
        x, y, z = coords
        argv += ["--damage-coords", f"{x},{y},{z}"]

    sys.argv = argv
    cli_sim.main()

print('\nAll scenarios finished.')
print('OUT_DB:', OUT_DB)

# --- [5a] Post-processing plots ---
import sqlite3
import pandas as pd
import sys
import cli.cli_post_processing as cli_post

with sqlite3.connect(OUT_DB) as conn:
    runs = pd.read_sql("SELECT id AS run_id FROM runs ORDER BY id", conn)

for run_id in runs["run_id"]:
    sys.argv = [
        "cli_post_processing.py",
        "--outdb", str(OUT_DB),
        "--figs-root", str(FIGS),
        "--make-plots", "yes",
        "--plot-mode", PLOT_MODE,
        "--run-id", str(int(run_id)),
    ]
    cli_post.main()

print("Figures written to:", FIGS)

# --- [5b] Entropy-based robustness index (pipeline function) ---

from pathlib import Path
import sqlite3
from pbm.post.robustness_plots import plot_entropy_robustness_index

# Paths (as defined above in the notebook)
outdb = OUT_DB
figs_dir = FIGS
figs_dir.mkdir(parents=True, exist_ok=True)

# Failure event definition(s)
S_LIM_VALUES = [2]  # Manuscript adverse states; extend for additional analyses.

# --- Scenario-based run selection (UI layer) ---
BASELINE_SCENARIO = 0

# All non-baseline scenarios
SCENARIO_KEYS = sorted(k for k in SCENARIOS if k != BASELINE_SCENARIO)

# Resolve run IDs from stored scenario IDs; do not assume insertion order.
with sqlite3.connect(outdb) as conn:
    run_rows = conn.execute(
        "SELECT id, scenario_id FROM runs ORDER BY id;"
    ).fetchall()
SCENARIO_TO_RUN_ID = {}
for run_id, scenario_id in run_rows:
    if scenario_id in SCENARIO_TO_RUN_ID:
        raise RuntimeError(f"Multiple runs found for scenario_id={scenario_id}.")
    SCENARIO_TO_RUN_ID[int(scenario_id)] = int(run_id)

# Derived DB-facing values (used by pipeline)
BASELINE_RUN_ID = SCENARIO_TO_RUN_ID[BASELINE_SCENARIO]
SCENARIO_RUN_IDS = [SCENARIO_TO_RUN_ID[k] for k in SCENARIO_KEYS]

# Legend labels (DB run_id -> human label)
scenario_labels = {
    SCENARIO_TO_RUN_ID[k]: SCENARIOS[k][0]
    for k in SCENARIO_KEYS
}

with sqlite3.connect(outdb) as conn:
    for s_lim in S_LIM_VALUES:
        plot_entropy_robustness_index(
            conn,
            baseline_run_id=BASELINE_RUN_ID,
            damaged_run_ids=SCENARIO_RUN_IDS,
            s_lim=int(s_lim),
            figs_dir=figs_dir,
            filename=f"entropy_robustness_index_Slim_{int(s_lim)}",
            formats=("pdf",),
            pi_min=-3.0,
            pi_max=3.0,
            n_grid=1001,
            scenario_labels=scenario_labels,
            coherence_mode="strict",
            mc_tolerance=0.0,
            plot_mode=PLOT_MODE
        )

print(f"Saved entropy robustness figures to: {figs_dir.resolve()}")
